# Olist Raw Data — Data Profiling

Systematic per-table profiling of the `raw` schema to surface statistical and content issues that constraint declaration cannot catch.

**For each table, we compute:**
- Row count
- Null count and percentage per column
- Distinct count for ID columns (verify uniqueness)
- Date ranges (min, max, range in days)
- Numeric distributions (min, max, mean, median, stdev)
- Top values for categorical columns

**Findings are appended to `docs/data_quality.md` as DQ-003 onwards.**

In [5]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Find .env by walking up from the notebook's cwd
def find_project_root(start: Path, marker: str = ".env") -> Path:
    for parent in [start, *start.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} walking up from {start}")

PROJECT_ROOT = find_project_root(Path.cwd())
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# Pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 100)

print(f"Project root: {PROJECT_ROOT}")
print("Setup complete. Engine ready.")

Project root: /Users/gowthamir/Projects/ecommerce-analytics-pipeline
Setup complete. Engine ready.


In [6]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'raw' 
        ORDER BY table_name;
    """))
    tables = [row[0] for row in result]

print(f"Tables in raw schema ({len(tables)}):")
for t in tables:
    print(f"  - {t}")

Tables in raw schema (9):
  - customers
  - geolocation
  - order_items
  - order_payments
  - order_reviews
  - orders
  - product_category_name_translation
  - products
  - sellers


---

## Table 1: `raw.customers`

99,441 rows. One row per *order-customer* (per Olist's identity model — a real human has a `customer_unique_id`, and gets a fresh `customer_id` per order).

**Profile checks:**
- Row count and null rates per column
- Distinct count of `customer_id` (should equal row count = PK is unique)
- Distinct count of `customer_unique_id` (gives us number of real humans)
- Top 10 states by customer count
- Distribution of cities (top 20)

In [7]:
df = pd.read_sql("SELECT * FROM raw.customers", engine)
print(f"Row count: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print()
print("Null counts and percentages:")
nulls = df.isnull().sum()
null_pcts = (df.isnull().mean() * 100).round(2)
null_summary = pd.DataFrame({"null_count": nulls, "null_pct": null_pcts})
null_summary

Row count: 99,441
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Null counts and percentages:


,null_count,null_pct
customer_id,0,0.0
customer_unique_id,0,0.0
customer_zip_code_prefix,0,0.0
customer_city,0,0.0
customer_state,0,0.0


In [8]:
total_rows = len(df)
distinct_customer_id = df["customer_id"].nunique()
distinct_unique_id = df["customer_unique_id"].nunique()

print(f"Total rows: {total_rows:,}")
print(f"Distinct customer_id: {distinct_customer_id:,}")
print(f"Distinct customer_unique_id: {distinct_unique_id:,}")
print()
print(f"Repeat-buyer ratio: {total_rows / distinct_unique_id:.4f}")
print(f"   ({total_rows - distinct_unique_id:,} customer_id rows belong to people who appear more than once)")

Total rows: 99,441
Distinct customer_id: 99,441
Distinct customer_unique_id: 96,096

Repeat-buyer ratio: 1.0348
   (3,345 customer_id rows belong to people who appear more than once)


In [9]:
print("Top 10 states by customer count:")
state_dist = df["customer_state"].value_counts().head(10)
state_pct = (df["customer_state"].value_counts(normalize=True) * 100).head(10).round(2)
state_summary = pd.DataFrame({
    "customer_count": state_dist,
    "pct_of_customers": state_pct
})
print(state_summary)

print()
print(f"Total distinct states: {df['customer_state'].nunique()}")
print(f"Total distinct cities: {df['customer_city'].nunique()}")

Top 10 states by customer count:
                customer_count  pct_of_customers
customer_state                                  
SP                       41746             41.98
RJ                       12852             12.92
MG                       11635             11.70
RS                        5466              5.50
PR                        5045              5.07
SC                        3637              3.66
BA                        3380              3.40
DF                        2140              2.15
ES                        2033              2.04
GO                        2020              2.03

Total distinct states: 27
Total distinct cities: 4119


### Customers — summary

- 99,441 rows, 5 columns, zero nulls
- `customer_id` is unique (verifies the PK declared on Day 2)
- 96,096 distinct `customer_unique_id` — repeat-buyer ratio of 1.035 (≈3.4% of orders are repeat purchases — unusually low for e-commerce)
- Geography heavily concentrated in the Southeast: SP+RJ+MG = 66.6% of customers
- 4,119 distinct cities — high cardinality dimension for later modeling

No data quality issues. Two business-context findings (low retention, geographic concentration) noted for the README narrative.

---

## Table 2: `raw.orders`

99,441 rows. Central transactional table. One row per order.

**Profile checks:**
- Row count and null rates per column (expect nulls in late-stage date columns)
- Distribution of `order_status` (the order funnel)
- Date range of purchase timestamps (dataset coverage)
- Date range checks for each date column
- Logical ordering: purchase → approved → carrier → customer (does it hold?)

In [10]:
orders = pd.read_sql("SELECT * FROM raw.orders", engine)
print(f"Row count: {len(orders):,}")
print(f"Columns: {list(orders.columns)}")
print()
print("Null counts and percentages:")
nulls = orders.isnull().sum()
null_pcts = (orders.isnull().mean() * 100).round(2)
null_summary = pd.DataFrame({"null_count": nulls, "null_pct": null_pcts})
null_summary

Row count: 99,441
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Null counts and percentages:


,null_count,null_pct
order_id,0,0.00
customer_id,0,0.00
order_status,0,0.00
order_purchase_timestamp,0,0.00
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98
order_estimated_delivery_date,0,0.00


In [11]:
print("Order status distribution:")
status_dist = orders["order_status"].value_counts()
status_pct = (orders["order_status"].value_counts(normalize=True) * 100).round(2)
status_summary = pd.DataFrame({
    "count": status_dist,
    "pct": status_pct
})
print(status_summary)
print()
print(f"Total distinct statuses: {orders['order_status'].nunique()}")

Order status distribution:
              count    pct
order_status              
delivered     96478  97.02
shipped        1107   1.11
canceled        625   0.63
unavailable     609   0.61
invoiced        314   0.32
processing      301   0.30
created           5   0.01
approved          2   0.00

Total distinct statuses: 8


In [12]:
# Cross-tabulation: order_status vs whether delivery date is null
orders["delivery_date_null"] = orders["order_delivered_customer_date"].isnull()

crosstab = pd.crosstab(
    orders["order_status"],
    orders["delivery_date_null"],
    margins=True,
    margins_name="TOTAL"
)
crosstab.columns = ["has_delivery_date", "no_delivery_date", "total"]
print("Order status vs. delivery date presence:")
print(crosstab)

Order status vs. delivery date presence:
              has_delivery_date  no_delivery_date  total
order_status                                            
approved                      0                 2      2
canceled                      6               619    625
created                       0                 5      5
delivered                 96470                 8  96478
invoiced                      0               314    314
processing                    0               301    301
shipped                       0              1107   1107
unavailable                   0               609    609
TOTAL                     96476              2965  99441


In [13]:
print("8 'delivered' orders with NULL delivery date")
delivered_no_date = orders[
    (orders["order_status"] == "delivered") & 
    (orders["order_delivered_customer_date"].isnull())
][["order_id", "order_status", "order_purchase_timestamp", 
   "order_approved_at", "order_delivered_carrier_date", 
   "order_delivered_customer_date"]]
print(delivered_no_date)

print()
print("6 'canceled' orders WITH delivery date ")
canceled_with_date = orders[
    (orders["order_status"] == "canceled") & 
    (orders["order_delivered_customer_date"].notnull())
][["order_id", "order_status", "order_purchase_timestamp", 
   "order_delivered_customer_date"]]
print(canceled_with_date)

8 'delivered' orders with NULL delivery date
                               order_id order_status order_purchase_timestamp   order_approved_at order_delivered_carrier_date order_delivered_customer_date
3002   2d1e2d5bf4dc7227b3bfebb81328c15f    delivered      2017-11-28 17:44:07 2017-11-28 17:56:40          2017-11-30 18:12:23                           NaT
20618  f5dd62b788049ad9fc0526e3ad11a097    delivered      2018-06-20 06:58:43 2018-06-20 07:19:05          2018-06-25 08:05:00                           NaT
43834  2ebdfc4f15f23b91474edf87475f108e    delivered      2018-07-01 17:05:11 2018-07-01 17:15:12          2018-07-03 13:57:00                           NaT
79263  e69f75a717d64fc5ecdfae42b2e8e086    delivered      2018-07-01 22:05:55 2018-07-01 22:15:14          2018-07-03 13:57:00                           NaT
82868  0d3268bad9b086af767785e3f0fc0133    delivered      2018-07-01 21:14:02 2018-07-01 21:29:54          2018-07-03 09:28:00                           NaT
92643  2d858f

### Orders — preliminary findings

- 99,441 rows, 8 columns, 5 of which are date columns
- Null pattern across delivery stages forms a clean staircase (0% purchase → 0.16% approved → 1.79% carrier → 2.98% customer date)
- 97% of orders are `delivered`; 8 distinct statuses; only 5 `created` and 2 `approved` (transient states)
- **DQ-003**: 8 orders marked `delivered` with no delivery date — status field is independent of timestamps
- **DQ-004**: 6 `canceled` orders have delivery timestamps — status overloads cancel + refund-after-delivery; clustered in Oct–Nov 2016

Status-vs-timestamp inconsistency means any time-based metric must trust timestamps, not status.

In [14]:
print("Date ranges across all 5 date columns:")
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
date_summary = pd.DataFrame({
    "min": [orders[c].min() for c in date_cols],
    "max": [orders[c].max() for c in date_cols],
    "non_null_count": [orders[c].notnull().sum() for c in date_cols],
}, index=date_cols)
print(date_summary)

Date ranges across all 5 date columns:
                                              min                 max  non_null_count
order_purchase_timestamp      2016-09-04 21:15:19 2018-10-17 17:30:18           99441
order_approved_at             2016-09-15 12:16:38 2018-09-03 17:40:06           99281
order_delivered_carrier_date  2016-10-08 10:34:01 2018-09-11 19:48:28           97658
order_delivered_customer_date 2016-10-11 13:46:32 2018-10-17 13:22:46           96476
order_estimated_delivery_date 2016-09-30 00:00:00 2018-11-12 00:00:00           99441


In [15]:
print("Date ordering checks (counting violations):")

# 1. purchase should be <= approved
v1 = orders[
    orders["order_approved_at"].notnull() & 
    (orders["order_purchase_timestamp"] > orders["order_approved_at"])
]
print(f"  Purchase > Approved: {len(v1)} violations")

# 2. approved should be <= carrier
v2 = orders[
    orders["order_approved_at"].notnull() & 
    orders["order_delivered_carrier_date"].notnull() &
    (orders["order_approved_at"] > orders["order_delivered_carrier_date"])
]
print(f"  Approved > Carrier: {len(v2)} violations")

# 3. carrier should be <= customer
v3 = orders[
    orders["order_delivered_carrier_date"].notnull() & 
    orders["order_delivered_customer_date"].notnull() &
    (orders["order_delivered_carrier_date"] > orders["order_delivered_customer_date"])
]
print(f"  Carrier > Customer: {len(v3)} violations")

# 4. purchase should be <= estimated delivery
v4 = orders[
    orders["order_purchase_timestamp"] > orders["order_estimated_delivery_date"]
]
print(f"  Purchase > Estimated delivery: {len(v4)} violations")

Date ordering checks (counting violations):
  Purchase > Approved: 0 violations
  Approved > Carrier: 1359 violations
  Carrier > Customer: 23 violations
  Purchase > Estimated delivery: 0 violations


In [17]:
print("Sample of 'Approved > Carrier' violations")
approved_after_carrier = orders[
    orders["order_approved_at"].notnull() & 
    orders["order_delivered_carrier_date"].notnull() &
    (orders["order_approved_at"] > orders["order_delivered_carrier_date"])
].copy()

approved_after_carrier["lag_hours"] = (
    approved_after_carrier["order_approved_at"] - 
    approved_after_carrier["order_delivered_carrier_date"]
).dt.total_seconds() / 3600

print(f"Total violations: {len(approved_after_carrier):,}")
print(f"\nLag (approved minus carrier) distribution:")
print(approved_after_carrier["lag_hours"].describe().round(2))

print(f"\nSample 5 rows:")
print(approved_after_carrier[
    ["order_id", "order_status", "order_purchase_timestamp", 
     "order_approved_at", "order_delivered_carrier_date", "lag_hours"]
].head())

Sample of 'Approved > Carrier' violations
Total violations: 1,359

Lag (approved minus carrier) distribution:
count    1359.00
mean       24.75
std       115.31
min         0.01
25%         1.42
50%        17.17
75%        25.96
max      4109.26
Name: lag_hours, dtype: float64

Sample 5 rows:
                             order_id order_status order_purchase_timestamp   order_approved_at order_delivered_carrier_date  lag_hours
15   dcb36b511fcac050b97cd5c05de84dc3    delivered      2018-06-07 19:03:12 2018-06-12 23:31:02          2018-06-11 14:54:00  32.617222
64   688052146432ef8253587b930b01a06d    delivered      2018-04-22 08:48:13 2018-04-24 18:25:22          2018-04-23 19:19:14  23.102222
199  58d4c4747ee059eeeb865b349b41f53a    delivered      2018-07-21 12:49:32 2018-07-26 23:31:53          2018-07-24 12:57:00  58.581389
210  412fccb2b44a99b36714bca3fef8ad7b    delivered      2018-07-22 22:30:05 2018-07-23 12:31:53          2018-07-23 12:24:00   0.131389
415  56a4ac10a4a8f2ba76935

### Orders — summary (complete)

- 99,441 rows across Sept 2016 → Oct 2018
- Null staircase across 5 date columns matches order funnel (0% → 0.16% → 1.79% → 2.98%, with estimated date 0%)
- 97% delivered; 8 distinct statuses
- Late Oct 2018 data is partial (approved_at max precedes purchase max by 6 weeks)

**4 data quality findings:**
- **DQ-003:** 8 delivered orders without delivery timestamp (status field is logical, not derived)
- **DQ-004:** 6 canceled orders with delivery timestamp (status overloads cancel + post-delivery refund)
- **DQ-005:** 1,359 orders where `approved_at` post-dates carrier event — column likely captures payment settlement, not order acceptance
- **DQ-006:** 23 orders where carrier timestamp post-dates customer delivery — clock skew

Orders table is informative but semantically messy. Status and timestamps must be interpreted independently.

---

## Table 3: `raw.order_items`

112,650 rows. Grain: one row per item-within-an-order. The line-item level of the data.

**Profile checks:**
- Null rates per column
- Items-per-order distribution
- `price` and `freight_value` distributions (look for outliers)
- Shipping limit date range

In [18]:
items = pd.read_sql("SELECT * FROM raw.order_items", engine)
print(f"Row count: {len(items):,}")
print(f"Columns: {list(items.columns)}")
print()
print("Null counts:")
nulls = items.isnull().sum()
null_pcts = (items.isnull().mean() * 100).round(2)
print(pd.DataFrame({"null_count": nulls, "null_pct": null_pcts}))

Row count: 112,650
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Null counts:
                     null_count  null_pct
order_id                      0       0.0
order_item_id                 0       0.0
product_id                    0       0.0
seller_id                     0       0.0
shipping_limit_date           0       0.0
price                         0       0.0
freight_value                 0       0.0


In [19]:
items_per_order = items.groupby("order_id").size()

print("Items per order distribution:")
print(items_per_order.describe().round(2))
print()
print("Distribution of order sizes:")
print(items_per_order.value_counts().sort_index().head(15))
print()
print(f"Orders with >5 items: {(items_per_order > 5).sum():,}")
print(f"Largest order: {items_per_order.max()} items")

Items per order distribution:
count    98666.00
mean         1.14
std          0.54
min          1.00
25%          1.00
50%          1.00
75%          1.00
max         21.00
dtype: float64

Distribution of order sizes:
1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8
11        4
12        5
13        1
14        2
15        2
Name: count, dtype: int64

Orders with >5 items: 256
Largest order: 21 items


In [20]:
# Find orders that have no items
all_order_ids = set(orders["order_id"])
order_ids_with_items = set(items["order_id"])
orders_no_items = all_order_ids - order_ids_with_items

print(f"Orders in raw.orders: {len(all_order_ids):,}")
print(f"Orders with items: {len(order_ids_with_items):,}")
print(f"Orders with NO items: {len(orders_no_items):,}")
print()

# Status breakdown of the item-less orders
no_item_orders_df = orders[orders["order_id"].isin(orders_no_items)]
print("Status distribution of item-less orders:")
print(no_item_orders_df["order_status"].value_counts())

Orders in raw.orders: 99,441
Orders with items: 98,666
Orders with NO items: 775

Status distribution of item-less orders:
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


In [21]:
print("Price distribution:")
print(items["price"].describe().round(2))
print()
print("Freight value distribution:")
print(items["freight_value"].describe().round(2))
print()
print("Top 10 most expensive items:")
print(items.nlargest(10, "price")[["order_id", "product_id", "price", "freight_value"]])
print()
print("Items with $0 price:")
print(f"  Count: {(items['price'] == 0).sum()}")
print()
print("Items with $0 freight:")
print(f"  Count: {(items['freight_value'] == 0).sum()}")

Price distribution:
count    112650.00
mean        120.65
std         183.63
min           0.85
25%          39.90
50%          74.99
75%         134.90
max        6735.00
Name: price, dtype: float64

Freight value distribution:
count    112650.00
mean         19.99
std          15.81
min           0.00
25%          13.08
50%          16.26
75%          21.15
max         409.68
Name: freight_value, dtype: float64

Top 10 most expensive items:
                                order_id                        product_id    price  freight_value
3556    0812eb902a67711a1cb742b3cdaa65ae  489ae2aa008f021502940f251d4cce7f  6735.00         194.31
112233  fefacc66af859508bf1a7934eab1e97f  69c590f7ffc7bf8db97190b6cb6ed62e  6729.00         193.21
107841  f5136e38d1a14a4dbd87dff67da82701  1bdf5e6731585cf01aa8169c7028d6ad  6499.00         227.66
74336   a96610ab360d42a2e5335a3998b4718a  a6492cc69376c469ab6f61d8f44de961  4799.00         151.34
11249   199af31afc78c699f0dbf71fb178d4d4  c3ed642d592594bb

In [22]:
zero_freight = items[items["freight_value"] == 0].copy()
print(f"Total $0 freight items: {len(zero_freight)}")
print()
print(f"Distinct sellers offering $0 freight: {zero_freight['seller_id'].nunique()}")
print(f"Distinct orders with $0 freight: {zero_freight['order_id'].nunique()}")
print()
print("Top 10 sellers by $0 freight item count:")
print(zero_freight["seller_id"].value_counts().head(10))
print()
print("Price distribution of $0 freight items:")
print(zero_freight["price"].describe().round(2))

Total $0 freight items: 383

Distinct sellers offering $0 freight: 9
Distinct orders with $0 freight: 339

Top 10 sellers by $0 freight item count:
seller_id
7d13fca15225358621be4086e1eb0964    158
955fee9216a65b617aa5c0531780ce60     99
1f50f920176fa81dab994f9023523100     56
4869f7a5dfa277a7dca6462dcf3b52b2     56
37be5a7c751166fbc5f8ccba4119e043      9
c826c40d7b19f62a09e2d7c5e7295ee2      2
bc2ac6b95e1accce9858528ee566c17e      1
cc419e0650a3c5ba77189a1882b7556a      1
8581055ce74af1daba164fdbd55a40de      1
Name: count, dtype: int64

Price distribution of $0 freight items:
count    383.0
mean      98.6
std       50.0
min       53.9
25%       69.9
50%       99.9
75%      106.9
max      712.9
Name: price, dtype: float64


In [23]:
# Join items with orders to compare shipping_limit vs purchase timestamp
items_with_purchase = items.merge(
    orders[["order_id", "order_purchase_timestamp"]],
    on="order_id",
    how="left"
)

# Calculate shipping lead time (days the seller has to ship)
items_with_purchase["ship_lead_days"] = (
    items_with_purchase["shipping_limit_date"] - 
    items_with_purchase["order_purchase_timestamp"]
).dt.total_seconds() / 86400

print("Shipping lead time (days from purchase to shipping deadline):")
print(items_with_purchase["ship_lead_days"].describe().round(2))
print()
print(f"Items with shipping_limit BEFORE purchase: {(items_with_purchase['ship_lead_days'] < 0).sum()}")
print(f"Items with shipping_limit > 30 days after purchase: {(items_with_purchase['ship_lead_days'] > 30).sum()}")

Shipping lead time (days from purchase to shipping deadline):
count    112650.00
mean          6.64
std           7.07
min           2.00
25%           5.01
50%           6.01
75%           7.19
max        1056.04
Name: ship_lead_days, dtype: float64

Items with shipping_limit BEFORE purchase: 0
Items with shipping_limit > 30 days after purchase: 239


In [24]:
print("Top 5 longest shipping lead times:")
print(items_with_purchase.nlargest(5, "ship_lead_days")[
    ["order_id", "order_purchase_timestamp", "shipping_limit_date", "ship_lead_days"]
])
print()
print("Distribution of 30+ day items:")
print(items_with_purchase[items_with_purchase["ship_lead_days"] > 30]["ship_lead_days"].describe().round(2))

Top 5 longest shipping lead times:
                               order_id order_purchase_timestamp shipping_limit_date  ship_lead_days
8643   13bdf405f961a6deec817d817f5c6624      2017-03-16 02:30:51 2020-02-05 03:30:51     1056.041667
68516  9c94a4ea2f7876660fa6f1b59b69c8e6      2017-03-14 19:23:22 2020-02-03 20:23:22     1056.041667
85729  c2bb89b5c1dd978d507284be78a04cb2      2017-05-23 22:28:36 2020-04-09 22:35:08     1052.004537
85730  c2bb89b5c1dd978d507284be78a04cb2      2017-05-23 22:28:36 2020-04-09 22:35:08     1052.004537
46557  69d126e78947276280838ee9361f5505      2017-03-30 15:23:23 2017-08-25 15:32:11      148.006111

Distribution of 30+ day items:
count     239.00
mean       56.27
std       130.95
min        30.01
25%        34.01
50%        36.01
75%        43.01
max      1056.04
Name: ship_lead_days, dtype: float64


### Order items — summary

- 112,650 rows, 7 columns, zero nulls
- Grain: one row per item per order; 90% of orders are single-item; max order has 21 items
- Price: median $74.99, max $6,735 (plausible)
- Freight: median $16.26, max $409.68 (plausible)

**3 data quality findings from order_items:**
- **DQ-007:** 775 orders have no item-level records (mostly explainable by status; 3 anomalous)
- **DQ-008:** 383 items with $0 freight cluster in 9 sellers — bundled-shipping pricing pattern
- **DQ-009:** 4 items have shipping_limit_date with apparent year-entry errors (off by 3 years)

---

## Table 4: `raw.order_payments`

103,886 rows. Grain: one row per payment installment within an order. Composite PK `(order_id, payment_sequential)`.

**Profile checks:**
- Null rates per column
- Payment type distribution
- Payment value distribution (zero, negative, outliers)
- Installment count distribution
- Payments-per-order pattern

In [25]:
payments = pd.read_sql("SELECT * FROM raw.order_payments", engine)
print(f"Row count: {len(payments):,}")
print(f"Columns: {list(payments.columns)}")
print()
print("Null counts:")
nulls = payments.isnull().sum()
null_pcts = (payments.isnull().mean() * 100).round(2)
print(pd.DataFrame({"null_count": nulls, "null_pct": null_pcts}))
print()
print("Payment type distribution:")
type_dist = payments["payment_type"].value_counts()
type_pct = (payments["payment_type"].value_counts(normalize=True) * 100).round(2)
print(pd.DataFrame({"count": type_dist, "pct": type_pct}))

Row count: 103,886
Columns: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Null counts:
                      null_count  null_pct
order_id                       0       0.0
payment_sequential             0       0.0
payment_type                   0       0.0
payment_installments           0       0.0
payment_value                  0       0.0

Payment type distribution:
              count    pct
payment_type              
credit_card   76795  73.92
boleto        19784  19.04
voucher        5775   5.56
debit_card     1529   1.47
not_defined       3   0.00


In [26]:
not_def = payments[payments["payment_type"] == "not_defined"]
print("=== 3 rows with payment_type = 'not_defined' ===")
print(not_def)
print()

# Join with orders to see context
not_def_with_orders = not_def.merge(
    orders[["order_id", "order_status", "order_purchase_timestamp"]],
    on="order_id",
    how="left"
)
print("With order context:")
print(not_def_with_orders[
    ["order_id", "payment_sequential", "payment_type", 
     "payment_installments", "payment_value", "order_status", "order_purchase_timestamp"]
])

=== 3 rows with payment_type = 'not_defined' ===
                               order_id  payment_sequential payment_type  payment_installments  payment_value
51280  4637ca194b6387e2d538dc89b124b0ee                   1  not_defined                     1            0.0
57413  00b1cb0320190ca0daa2c88b35206009                   1  not_defined                     1            0.0
94427  c8c528189310eaa44a745b8d9d26908b                   1  not_defined                     1            0.0

With order context:
                           order_id  payment_sequential payment_type  payment_installments  payment_value order_status order_purchase_timestamp
0  4637ca194b6387e2d538dc89b124b0ee                   1  not_defined                     1            0.0     canceled      2018-09-03 14:14:25
1  00b1cb0320190ca0daa2c88b35206009                   1  not_defined                     1            0.0     canceled      2018-08-28 15:26:39
2  c8c528189310eaa44a745b8d9d26908b                   1  n

In [27]:
print("Payment value distribution:")
print(payments["payment_value"].describe().round(2))
print()
print(f"Payments with $0 value: {(payments['payment_value'] == 0).sum()}")
print(f"Payments with negative value: {(payments['payment_value'] < 0).sum()}")
print()
print("Top 10 largest payments:")
print(payments.nlargest(10, "payment_value")[
    ["order_id", "payment_sequential", "payment_type", 
     "payment_installments", "payment_value"]
])

Payment value distribution:
count    103886.00
mean        154.10
std         217.49
min           0.00
25%          56.79
50%         100.00
75%         171.84
max       13664.08
Name: payment_value, dtype: float64

Payments with $0 value: 9
Payments with negative value: 0

Top 10 largest payments:
                               order_id  payment_sequential payment_type  payment_installments  payment_value
52108  03caa2c082116e1d31e67e9ae3700499                   1  credit_card                     1       13664.08
34370  736e1922ae60d0d6a89247b851902527                   1       boleto                     1        7274.88
41419  0812eb902a67711a1cb742b3cdaa65ae                   1  credit_card                     8        6929.31
49581  fefacc66af859508bf1a7934eab1e97f                   1       boleto                     1        6922.21
85539  f5136e38d1a14a4dbd87dff67da82701                   1       boleto                     1        6726.66
62409  2cc9089445046817a7539d90805e6e5a

In [28]:
zero_value = payments[(payments["payment_value"] == 0) & (payments["payment_type"] != "not_defined")]
print(f"Zero-value payments with defined payment type: {len(zero_value)}")
print()

# Show with order context
zero_value_context = zero_value.merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="left"
)

# Also, check: does this order have OTHER payment rows that are nonzero?
order_payment_totals = payments.groupby("order_id")["payment_value"].sum().reset_index()
order_payment_totals.columns = ["order_id", "order_total_payment"]

zero_with_total = zero_value_context.merge(order_payment_totals, on="order_id", how="left")

print("Zero-value rows with their full order context:")
print(zero_with_total[
    ["order_id", "payment_sequential", "payment_type",
     "payment_installments", "payment_value", 
     "order_status", "order_total_payment"]
])

Zero-value payments with defined payment type: 6

Zero-value rows with their full order context:
                           order_id  payment_sequential payment_type  payment_installments  payment_value order_status  order_total_payment
0  8bcbe01d44d147f901cd3192671144db                   4      voucher                     1            0.0    delivered                74.16
1  fa65dad1b0e818e3ccc5cb0e39231352                  14      voucher                     1            0.0      shipped               457.99
2  6ccb433e00daae1283ccc956189c82ae                   4      voucher                     1            0.0    delivered               122.04
3  45ed6e85398a87c253db47c2d9f48216                   3      voucher                     1            0.0    delivered                71.14
4  fa65dad1b0e818e3ccc5cb0e39231352                  13      voucher                     1            0.0      shipped               457.99
5  b23878b3e8eb4d25a158f57d96331b18                   4      vo

In [29]:
print("Installment count distribution:")
inst_dist = payments["payment_installments"].value_counts().sort_index()
inst_pct = (payments["payment_installments"].value_counts(normalize=True) * 100).sort_index().round(2)
inst_summary = pd.DataFrame({"count": inst_dist, "pct": inst_pct})
print(inst_summary)
print()
print(f"Max installments: {payments['payment_installments'].max()}")
print(f"Min installments: {payments['payment_installments'].min()}")
print()
print(f"Mean installments (credit_card only): "
      f"{payments[payments['payment_type'] == 'credit_card']['payment_installments'].mean():.2f}")

Installment count distribution:
                      count    pct
payment_installments              
0                         2   0.00
1                     52546  50.58
2                     12413  11.95
3                     10461  10.07
4                      7098   6.83
5                      5239   5.04
6                      3920   3.77
7                      1626   1.57
8                      4268   4.11
9                       644   0.62
10                     5328   5.13
11                       23   0.02
12                      133   0.13
13                       16   0.02
14                       15   0.01
15                       74   0.07
16                        5   0.00
17                        8   0.01
18                       27   0.03
20                       17   0.02
21                        3   0.00
22                        1   0.00
23                        1   0.00
24                       18   0.02

Max installments: 24
Min installments: 0

Mean installmen

In [30]:
zero_inst = payments[payments["payment_installments"] == 0]
zero_inst_context = zero_inst.merge(
    orders[["order_id", "order_status", "order_purchase_timestamp"]],
    on="order_id",
    how="left"
)
print("=== 2 rows with payment_installments = 0 ===")
print(zero_inst_context[
    ["order_id", "payment_sequential", "payment_type",
     "payment_installments", "payment_value",
     "order_status", "order_purchase_timestamp"]
])

=== 2 rows with payment_installments = 0 ===
                           order_id  payment_sequential payment_type  payment_installments  payment_value order_status order_purchase_timestamp
0  744bade1fcf9ff3f31d860ace076d422                   2  credit_card                     0          58.69    delivered      2018-04-22 11:34:42
1  1a57108394169c0b47d8f876acc9ba2d                   2  credit_card                     0         129.94    delivered      2018-05-15 16:25:14


In [31]:
payments_per_order = payments.groupby("order_id").size()

print("Payments per order distribution:")
print(payments_per_order.describe().round(2))
print()
print("Top of distribution:")
print(payments_per_order.value_counts().sort_index().head(15))
print()
print(f"Orders with 1 payment row: {(payments_per_order == 1).sum():,}")
print(f"Orders with >5 payment rows: {(payments_per_order > 5).sum():,}")
print(f"Orders with >20 payment rows: {(payments_per_order > 20).sum():,}")
print(f"Max payments-per-order: {payments_per_order.max()}")

Payments per order distribution:
count    99440.00
mean         1.04
std          0.38
min          1.00
25%          1.00
50%          1.00
75%          1.00
max         29.00
dtype: float64

Top of distribution:
1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
Name: count, dtype: int64

Orders with 1 payment row: 96,479
Orders with >5 payment rows: 118
Orders with >20 payment rows: 4
Max payments-per-order: 29


In [32]:
all_order_ids = set(orders["order_id"])
order_ids_with_payments = set(payments["order_id"])
orders_no_payment = all_order_ids - order_ids_with_payments

print(f"Orders with no payment rows: {len(orders_no_payment)}")
print()
no_pay_df = orders[orders["order_id"].isin(orders_no_payment)]
print(no_pay_df[["order_id", "order_status", "order_purchase_timestamp"]])

Orders with no payment rows: 1

                               order_id order_status order_purchase_timestamp
30710  bfbd0f9bdef84302105ad712db648a6c    delivered      2016-09-15 12:16:38


### Order payments — summary

- 103,886 rows, 5 columns, zero nulls
- Payment type mix: credit_card (74%), boleto (19%), voucher (5.6%), debit (1.5%), not_defined (0.003%)
- Median payment $100, max $13,664 (multi-item bulk purchases)
- Mean installments for credit_card: 3.51 — meaningful "parcelado" usage
- 97% of orders have a single payment row; long tail extends to 29 payment rows (voucher splits)

**3 data quality findings:**
- **DQ-010:** 3 `not_defined` payment_type rows (canceled-order placeholders) + addendum on 6 zero-value voucher rows (accounting artifacts)
- **DQ-011:** 2 credit_card payments with `installments=0` (data entry errors)
- **DQ-012:** 1 delivered order with no payment record (launch-era anomaly)

---

## Table 5: `raw.order_reviews`

99,224 rows. Composite grain: one row per (review, order) pair (per DQ-001).

**Profile checks:**
- Null rates per column (expect high nulls in comment fields)
- Review score distribution (1–5 scale)
- Response lag (review_creation → review_answer)
- Reviews per order (most have 1, some have multiple per DQ-001)
- Date range

In [33]:
reviews = pd.read_sql("SELECT * FROM raw.order_reviews", engine)
print(f"Row count: {len(reviews):,}")
print(f"Columns: {list(reviews.columns)}")
print()
print("Null counts:")
nulls = reviews.isnull().sum()
null_pcts = (reviews.isnull().mean() * 100).round(2)
print(pd.DataFrame({"null_count": nulls, "null_pct": null_pcts}))
print()
print("Review score distribution:")
score_dist = reviews["review_score"].value_counts().sort_index()
score_pct = (reviews["review_score"].value_counts(normalize=True) * 100).sort_index().round(2)
print(pd.DataFrame({"count": score_dist, "pct": score_pct}))

Row count: 99,224
Columns: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

Null counts:
                         null_count  null_pct
review_id                         0      0.00
order_id                          0      0.00
review_score                      0      0.00
review_comment_title          87656     88.34
review_comment_message        58247     58.70
review_creation_date              0      0.00
review_answer_timestamp           0      0.00

Review score distribution:
              count    pct
review_score              
1             11424  11.51
2              3151   3.18
3              8179   8.24
4             19142  19.29
5             57328  57.78


In [34]:
reviews["response_lag_hours"] = (
    reviews["review_answer_timestamp"] - reviews["review_creation_date"]
).dt.total_seconds() / 3600

print("Response lag distribution (hours from creation to answer):")
print(reviews["response_lag_hours"].describe().round(2))
print()
print(f"Reviews with negative lag (answered before created): "
      f"{(reviews['response_lag_hours'] < 0).sum()}")
print()
print(f"Reviews with same-day response (<24h): "
      f"{(reviews['response_lag_hours'] < 24).sum():,} "
      f"({(reviews['response_lag_hours'] < 24).mean() * 100:.1f}%)")

Response lag distribution (hours from creation to answer):
count    99224.00
mean        75.58
std        237.36
min          2.14
25%         24.12
50%         40.20
75%         74.49
max      12448.78
Name: response_lag_hours, dtype: float64

Reviews with negative lag (answered before created): 0

Reviews with same-day response (<24h): 24,361 (24.6%)


In [35]:
reviews_per_order = reviews.groupby("order_id").size()

print("Reviews-per-order distribution:")
print(reviews_per_order.value_counts().sort_index())
print()
print(f"Orders with multiple reviews: {(reviews_per_order > 1).sum()}")
print(f"Max reviews on one order: {reviews_per_order.max()}")
print()
# Also: how many orders have NO review?
all_order_ids = set(orders["order_id"])
order_ids_with_reviews = set(reviews["order_id"])
orders_no_review = all_order_ids - order_ids_with_reviews
print(f"Orders with NO review: {len(orders_no_review):,} "
      f"({len(orders_no_review) / len(all_order_ids) * 100:.1f}%)")

Reviews-per-order distribution:
1    98126
2      543
3        4
Name: count, dtype: int64

Orders with multiple reviews: 547
Max reviews on one order: 3

Orders with NO review: 768 (0.8%)


In [37]:
# Inspect one order with 3 reviews
print("Orders with 3 reviews — sample inspection")
three_review_orders = reviews_per_order[reviews_per_order == 3].index.tolist()
for oid in three_review_orders[:2]:  # First 2 of 4
    print(f"\nOrder {oid}:")
    print(reviews[reviews["order_id"] == oid][
        ["review_id", "review_score", "review_creation_date", "review_answer_timestamp"]
    ])

# Status distribution of orders with no review
print("\n\n=== Status distribution of 768 orders with no review ===")
no_review_orders = orders[orders["order_id"].isin(orders_no_review)]
print(no_review_orders["order_status"].value_counts())

Orders with 3 reviews — sample inspection

Order 03c939fd7fd3b38f8485a0f95798f1f6:
                              review_id  review_score review_creation_date review_answer_timestamp
8273   b04ed893318da5b863e878cd3d0511df             3           2018-03-20     2018-03-21 02:28:23
51527  f4bb9d6dd4fb6dcc2298f0e7b17b8e1e             4           2018-03-29     2018-03-30 00:29:09
69438  405eb2ea45e1dbe2662541ae5b47e2aa             3           2018-03-06     2018-03-06 19:50:32

Order 8e17072ec97ce29f0e1f111e598b0c85:
                              review_id  review_score review_creation_date review_answer_timestamp
44694  67c2557eb0bd72e3ece1e03477c9dff5             1           2018-04-07     2018-04-08 22:48:27
64510  2d6ac45f859465b5c185274a1c929637             1           2018-04-07     2018-04-07 21:13:05
92300  6e4c4086d9611ae4cc0cc65a262751fe             1           2018-04-14     2018-04-16 11:37:31


=== Status distribution of 768 orders with no review ===
order_status
delivered   

### Order reviews — summary

- 99,224 rows; composite grain `(review_id, order_id)`
- Comment fields heavily null: title 88%, message 59% (product design, not data issue)
- Score distribution heavily skewed positive: mean ~4.1, 78% are 4–5 stars, 12% are 1–2 stars
- Response lag: median 40 hours (~1.7 days); 24.6% same-day response
- 99.3% of delivered orders generated at least one review — exceptionally high opt-in rate

**1 new data quality finding:**
- **DQ-013:** 547 orders have multiple review submissions — inverse pattern of DQ-001; together they form a true many-to-many relationship between reviews and orders

Reviews are clean data-wise; the structural complexity is the (review_id, order_id) many-to-many we now have on both axes.

---

## Table 6: `raw.products`

32,951 rows. Catalog dimension. PK: `product_id`.

**Profile checks:**
- Null rates per column (610 known nulls in category, plus the typo'd `lenght` columns)
- Weight distribution (look for impossible values)
- Volumetric dimensions (length, height, width)
- Photo count distribution
- Top product categories
- Co-null behavior: when category is null, are physical attributes also null?

In [38]:
products = pd.read_sql("SELECT * FROM raw.products", engine)
print(f"Row count: {len(products):,}")
print(f"Columns: {list(products.columns)}")
print()
print("Null counts:")
nulls = products.isnull().sum()
null_pcts = (products.isnull().mean() * 100).round(2)
print(pd.DataFrame({"null_count": nulls, "null_pct": null_pcts}))

Row count: 32,951
Columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Null counts:
                            null_count  null_pct
product_id                           0      0.00
product_category_name              610      1.85
product_name_lenght                610      1.85
product_description_lenght         610      1.85
product_photos_qty                 610      1.85
product_weight_g                     2      0.01
product_length_cm                    2      0.01
product_height_cm                    2      0.01
product_width_cm                     2      0.01


In [39]:
# Verify: do the same 610 rows have null across all metadata fields?
metadata_nulls = products[
    products["product_category_name"].isnull() &
    products["product_name_lenght"].isnull() &
    products["product_description_lenght"].isnull() &
    products["product_photos_qty"].isnull()
]
print(f"Rows with ALL metadata fields null: {len(metadata_nulls)}")
print()

# And the 2 rows with null physical attributes - are they ALSO in the 610?
physical_nulls = products[
    products["product_weight_g"].isnull() &
    products["product_length_cm"].isnull() &
    products["product_height_cm"].isnull() &
    products["product_width_cm"].isnull()
]
print(f"Rows with ALL physical fields null: {len(physical_nulls)}")
print()

# Intersection: rows null in BOTH metadata and physical
both_null = products[
    products["product_category_name"].isnull() &
    products["product_weight_g"].isnull()
]
print(f"Rows null in BOTH metadata AND physical: {len(both_null)}")
print()
print("Sample physical-null rows:")
print(physical_nulls[["product_id", "product_category_name", 
                       "product_weight_g", "product_length_cm", 
                       "product_height_cm", "product_width_cm"]])

Rows with ALL metadata fields null: 610

Rows with ALL physical fields null: 2

Rows null in BOTH metadata AND physical: 1

Sample physical-null rows:
                             product_id product_category_name  product_weight_g  product_length_cm  product_height_cm  product_width_cm
8578   09ff539a621711667c43eba6a3bd8466                 bebes               NaN                NaN                NaN               NaN
18851  5eb564652db742ff8f28759cd8d2652a                   NaN               NaN                NaN                NaN               NaN


In [40]:
print("Weight distribution (grams):")
print(products["product_weight_g"].describe().round(2))
print()
print("Top 5 heaviest products:")
print(products.nlargest(5, "product_weight_g")[
    ["product_id", "product_category_name", "product_weight_g",
     "product_length_cm", "product_height_cm", "product_width_cm"]
])
print()
print("Length distribution (cm):")
print(products["product_length_cm"].describe().round(2))
print()
print("Height distribution (cm):")
print(products["product_height_cm"].describe().round(2))
print()
print("Width distribution (cm):")
print(products["product_width_cm"].describe().round(2))
print()
print("Photo count distribution:")
print(products["product_photos_qty"].value_counts().sort_index().head(15))

Weight distribution (grams):
count    32949.00
mean      2276.47
std       4282.04
min          0.00
25%        300.00
50%        700.00
75%       1900.00
max      40425.00
Name: product_weight_g, dtype: float64

Top 5 heaviest products:
                             product_id  product_category_name  product_weight_g  product_length_cm  product_height_cm  product_width_cm
25166  26644690fde745fc4654719c3904e1db        cama_mesa_banho           40425.0               13.0               65.0              28.0
344    d0877f0094337c414d23f5a3c7bad20c      moveis_escritorio           30000.0               50.0               50.0              30.0
509    53f92b0474f91fcb5bd188c6a8075c38  utilidades_domesticas           30000.0               76.0               51.0              51.0
955    ceeba7d5636e59173cc5f484e913db3d                    NaN           30000.0               65.0               65.0              65.0
1159   f97ad9066c718a6cef93dfcf253d3e0d       moveis_decoracao           3000

In [41]:
# Zero-weight products
zero_weight = products[products["product_weight_g"] == 0]
print(f"Products with 0g weight: {len(zero_weight)}")
print()
if len(zero_weight) > 0:
    print("Zero-weight products with their dimensions:")
    print(zero_weight[["product_id", "product_category_name", 
                       "product_weight_g", "product_length_cm",
                       "product_height_cm", "product_width_cm"]])

print()
# Products with implausibly small dimensions but nonzero weight
print("Products weighing >10kg with their dimensions:")
print(f"  Count: {(products['product_weight_g'] > 10000).sum()}")

Products with 0g weight: 4

Zero-weight products with their dimensions:
                             product_id product_category_name  product_weight_g  product_length_cm  product_height_cm  product_width_cm
9769   81781c0fed9fe1ad6e8c81fca1e1cb08       cama_mesa_banho               0.0               30.0               25.0              30.0
13683  8038040ee2a71048d4bdbbdc985b69ab       cama_mesa_banho               0.0               30.0               25.0              30.0
14997  36ba42dd187055e1fbe943b2d11430ca       cama_mesa_banho               0.0               30.0               25.0              30.0
32079  e673e90efa65a5409ff4196c038bb5af       cama_mesa_banho               0.0               30.0               25.0              30.0

Products weighing >10kg with their dimensions:
  Count: 1891


In [42]:
print("Top 15 product categories by count:")
cat_dist = products["product_category_name"].value_counts().head(15)
cat_pct = (products["product_category_name"].value_counts(normalize=True) * 100).head(15).round(2)
print(pd.DataFrame({"count": cat_dist, "pct": cat_pct}))
print()
print(f"Total distinct categories: {products['product_category_name'].nunique()}")
print(f"Products with NULL category: {products['product_category_name'].isnull().sum()}")

Top 15 product categories by count:
                             count   pct
product_category_name                   
cama_mesa_banho               3029  9.37
esporte_lazer                 2867  8.86
moveis_decoracao              2657  8.22
beleza_saude                  2444  7.56
utilidades_domesticas         2335  7.22
automotivo                    1900  5.87
informatica_acessorios        1639  5.07
brinquedos                    1411  4.36
relogios_presentes            1329  4.11
telefonia                     1134  3.51
bebes                          919  2.84
perfumaria                     868  2.68
papelaria                      849  2.63
fashion_bolsas_e_acessorios    849  2.63
cool_stuff                     789  2.44

Total distinct categories: 73
Products with NULL category: 610


### Products — summary

- 32,951 rows. PK on `product_id`.
- Two distinct null patterns: 610 ghost products (all metadata fields null), 2 products with all physical attributes null (1 overlapping with ghost group)
- 73 distinct categories; top 5 cover 41%; no extreme skew
- Weight: median 700g, max 40.4kg (heavy furniture/linens). 1,891 products >10kg.
- Dimensions: median 25×13×20 cm; all within plausible ranges
- Photos: most products have 1–3; max 15
- Source typo preserved: `product_name_lenght` and `product_description_lenght` (sic)

**1 new data quality finding:**
- **DQ-014:** 2 products with all-null physical attributes (one is a "ghost" product from DQ-002; one has a valid category but missing dimensions)
- **DQ-015:** 4 products with 0g weight, identical dimensions — copied listing template with placeholder values

---

## Table 7: `raw.sellers`

3,095 rows. Sellers dimension. PK: `seller_id`. Mirror structure to `customers`.

**Profile checks:**
- Null rates per column
- Geographic concentration (state distribution — compare to customers)
- City distribution
- Cross-reference: how many sellers actually appear in `order_items`?

In [43]:
sellers = pd.read_sql("SELECT * FROM raw.sellers", engine)
print(f"Row count: {len(sellers):,}")
print(f"Columns: {list(sellers.columns)}")
print()
print("Null counts:")
nulls = sellers.isnull().sum()
null_pcts = (sellers.isnull().mean() * 100).round(2)
print(pd.DataFrame({"null_count": nulls, "null_pct": null_pcts}))
print()
print("Top 10 states by seller count:")
state_dist = sellers["seller_state"].value_counts().head(10)
state_pct = (sellers["seller_state"].value_counts(normalize=True) * 100).head(10).round(2)
print(pd.DataFrame({"count": state_dist, "pct": state_pct}))
print()
print(f"Total distinct states: {sellers['seller_state'].nunique()}")
print(f"Total distinct cities: {sellers['seller_city'].nunique()}")

Row count: 3,095
Columns: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Null counts:
                        null_count  null_pct
seller_id                        0       0.0
seller_zip_code_prefix           0       0.0
seller_city                      0       0.0
seller_state                     0       0.0

Top 10 states by seller count:
              count    pct
seller_state              
SP             1849  59.74
PR              349  11.28
MG              244   7.88
SC              190   6.14
RJ              171   5.53
RS              129   4.17
GO               40   1.29
DF               30   0.97
ES               23   0.74
BA               19   0.61

Total distinct states: 23
Total distinct cities: 611


In [44]:
sellers_in_items = items["seller_id"].nunique()
all_sellers = sellers["seller_id"].nunique()
inactive_sellers = all_sellers - sellers_in_items

print(f"Total registered sellers: {all_sellers:,}")
print(f"Sellers with at least one order_item: {sellers_in_items:,}")
print(f"Sellers with ZERO sales: {inactive_sellers:,} ({inactive_sellers/all_sellers*100:.1f}%)")
print()

# Sales-per-seller distribution
sales_per_seller = items.groupby("seller_id").size()
print("Items-sold-per-seller distribution:")
print(sales_per_seller.describe().round(2))
print()

# How concentrated is the seller revenue?
print("Top sellers by item count:")
top_sellers = sales_per_seller.sort_values(ascending=False).head(10)
print(top_sellers)
print()
print(f"Top 10 sellers account for: "
      f"{top_sellers.sum() / sales_per_seller.sum() * 100:.1f}% of all items sold")
print(f"Top 100 sellers account for: "
      f"{sales_per_seller.sort_values(ascending=False).head(100).sum() / sales_per_seller.sum() * 100:.1f}%")

Total registered sellers: 3,095
Sellers with at least one order_item: 3,095
Sellers with ZERO sales: 0 (0.0%)

Items-sold-per-seller distribution:
count    3095.00
mean       36.40
std       119.19
min         1.00
25%         2.00
50%         8.00
75%        24.00
max      2033.00
dtype: float64

Top sellers by item count:
seller_id
6560211a19b47992c3666cc44a7e94c0    2033
4a3ca9315b744ce9f8e9374361493884    1987
1f50f920176fa81dab994f9023523100    1931
cc419e0650a3c5ba77189a1882b7556a    1775
da8622b14eb17ae2831f4ac5b9dab84a    1551
955fee9216a65b617aa5c0531780ce60    1499
1025f0e2d44d7041d6cf58b6550e0bfa    1428
7c67e1448b00f6e969d365cea6b010ab    1364
ea8482cd71df3c1969d7b9473ff13abc    1203
7a67c85e85bb2ce8582c35f2203ad736    1171
dtype: int64

Top 10 sellers account for: 14.2% of all items sold
Top 100 sellers account for: 44.9%


### Sellers — summary

- 3,095 rows, 4 columns, zero nulls. Clean.
- Geographic concentration **tighter than customers**: SP 60% of sellers vs 42% of customers; top 3 states = 79% (vs 67% for customers)
- 611 distinct cities, 23 distinct states (fewer than customers; supply side narrower than demand side)
- **100% activated** — every registered seller sold at least one item
- Power-law sales distribution: median 8 items/seller, max 2,033 items/seller; top 100 sellers = 45% of volume

No data quality findings. Two business insights for README: supply-side geographic concentration in SP, and Pareto-like seller volume distribution.

---

## Table 8: `raw.product_category_name_translation`

71 rows. Lookup table mapping Portuguese category names to English. Already partially documented in DQ-002.

**Profile checks:**
- Null rates
- Distinct counts (any duplicates on either column?)
- Sample translations

In [45]:
trans = pd.read_sql("SELECT * FROM raw.product_category_name_translation", engine)
print(f"Row count: {len(trans):,}")
print(f"Columns: {list(trans.columns)}")
print()
print("Null counts:")
nulls = trans.isnull().sum()
null_pcts = (trans.isnull().mean() * 100).round(2)
print(pd.DataFrame({"null_count": nulls, "null_pct": null_pcts}))
print()
print(f"Distinct Portuguese names: {trans['product_category_name'].nunique()}")
print(f"Distinct English names: {trans['product_category_name_english'].nunique()}")
print()
print("Sample translations:")
print(trans.head(10))

Row count: 71
Columns: ['product_category_name', 'product_category_name_english']

Null counts:
                               null_count  null_pct
product_category_name                   0       0.0
product_category_name_english           0       0.0

Distinct Portuguese names: 71
Distinct English names: 71

Sample translations:
    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto
3         cama_mesa_banho                bed_bath_table
4        moveis_decoracao               furniture_decor
5           esporte_lazer                sports_leisure
6              perfumaria                     perfumery
7   utilidades_domesticas                    housewares
8               telefonia                     telephony
9      relogios_presentes                 watches_gifts


### Product category translation — summary

- 71 rows, 2 columns, zero nulls. Clean.
- One-to-one mapping: 71 distinct Portuguese names → 71 distinct English names. No translation collisions.
- Translations are semantically reasonable on sample inspection.
- **Coverage gap already documented in DQ-002**: 2 product categories (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`) exist in `products` but not here.

No new data quality findings. This table is internally clean; the issue is its incompleteness relative to `products`.

---

## Table 9: `raw.geolocation`

1,000,163 rows. No PK (deliberately — multi-row-per-zip-prefix). Reference table mapping zip prefixes to geographic coordinates.

**Profile checks:**
- Null rates
- Distinct zip prefixes vs total rows (quantify multi-grain)
- Distinct cities/states
- Lat/long bounding box (sanity check vs Brazil's geography)
- Cross-reference: are all customer/seller zip prefixes covered?


In [47]:
geo = pd.read_sql("SELECT * FROM raw.geolocation", engine)
print(f"Row count: {len(geo):,}")
print(f"Columns: {list(geo.columns)}")
print()
print("Null counts:")
nulls = geo.isnull().sum()
null_pcts = (geo.isnull().mean() * 100).round(2)
print(pd.DataFrame({"null_count": nulls, "null_pct": null_pcts}))
print()
print(f"Distinct zip prefixes: {geo['geolocation_zip_code_prefix'].nunique():,}")
print(f"Distinct cities: {geo['geolocation_city'].nunique():,}")
print(f"Distinct states: {geo['geolocation_state'].nunique():,}")
print()
print("Rows-per-zip-prefix distribution:")
rows_per_zip = geo.groupby("geolocation_zip_code_prefix").size()
print(rows_per_zip.describe().round(2))
print()
print(f"Zip prefixes with >100 rows: {(rows_per_zip > 100).sum():,}")
print(f"Zip prefixes with >1000 rows: {(rows_per_zip > 1000).sum():,}")
print(f"Max rows for a single zip prefix: {rows_per_zip.max():,}")

Row count: 1,000,163
Columns: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Null counts:
                             null_count  null_pct
geolocation_zip_code_prefix           0       0.0
geolocation_lat                       0       0.0
geolocation_lng                       0       0.0
geolocation_city                      0       0.0
geolocation_state                     0       0.0

Distinct zip prefixes: 19,015
Distinct cities: 8,011
Distinct states: 27

Rows-per-zip-prefix distribution:
count    19015.00
mean        52.60
std         72.06
min          1.00
25%         10.00
50%         29.00
75%         66.50
max       1146.00
dtype: float64

Zip prefixes with >100 rows: 2,725
Zip prefixes with >1000 rows: 2
Max rows for a single zip prefix: 1,146


In [48]:
print("Latitude distribution:")
print(geo["geolocation_lat"].describe().round(4))
print()
print("Longitude distribution:")
print(geo["geolocation_lng"].describe().round(4))
print()

# Brazil bounding box: lat -34 to +5, lng -74 to -34
out_of_brazil = geo[
    (geo["geolocation_lat"] < -34) | (geo["geolocation_lat"] > 5) |
    (geo["geolocation_lng"] < -74) | (geo["geolocation_lng"] > -34)
]
print(f"Rows with coordinates outside Brazil's bounding box: {len(out_of_brazil):,}")
print()
if len(out_of_brazil) > 0:
    print("Sample out-of-bounds rows:")
    print(out_of_brazil[
        ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng",
         "geolocation_city", "geolocation_state"]
    ].head(10))

Latitude distribution:
count    1.000163e+06
mean    -2.117620e+01
std      5.715900e+00
min     -3.660540e+01
25%     -2.360350e+01
50%     -2.291940e+01
75%     -1.997960e+01
max      4.506590e+01
Name: geolocation_lat, dtype: float64

Longitude distribution:
count    1.000163e+06
mean    -4.639050e+01
std      4.269700e+00
min     -1.014668e+02
25%     -4.857320e+01
50%     -4.663790e+01
75%     -4.376770e+01
max      1.211054e+02
Name: geolocation_lng, dtype: float64

Rows with coordinates outside Brazil's bounding box: 42

Sample out-of-bounds rows:
       geolocation_zip_code_prefix  geolocation_lat  geolocation_lng         geolocation_city geolocation_state
387565                       18243        28.008978       -15.536867  bom retiro da esperanca                SP
513631                       28165        41.614052        -8.411675      vila nova de campos                RJ
513643                       28155       -34.586422       -58.732101              santa maria          

In [49]:
# For each zip prefix, count distinct city names
city_variants = geo.groupby("geolocation_zip_code_prefix")["geolocation_city"].nunique()

print("Distinct city names per zip prefix:")
print(city_variants.describe().round(2))
print()
print(f"Zip prefixes with multiple city names: {(city_variants > 1).sum():,} ({(city_variants > 1).mean()*100:.1f}%)")
print(f"Max city variants on one zip: {city_variants.max()}")
print()

# Pick one of the worst offenders and inspect
worst_zip = city_variants.idxmax()
print(f"\n=== Zip prefix {worst_zip} — {city_variants.max()} distinct city names ===")
worst_examples = geo[geo["geolocation_zip_code_prefix"] == worst_zip]["geolocation_city"].value_counts()
print(worst_examples)

Distinct city names per zip prefix:
count    19015.00
mean         1.47
std          0.54
min          1.00
25%          1.00
50%          1.00
75%          2.00
max          5.00
Name: geolocation_city, dtype: float64

Zip prefixes with multiple city names: 8,556 (45.0%)
Max city variants on one zip: 5


=== Zip prefix 06900 — 5 distinct city names ===
geolocation_city
embu-guacu    369
embu-guaçu     46
embu guaçu     21
embu guacu      7
embuguacu       4
Name: count, dtype: int64


In [50]:
customer_zips = set(pd.read_sql("SELECT DISTINCT customer_zip_code_prefix FROM raw.customers", engine)["customer_zip_code_prefix"])
seller_zips = set(pd.read_sql("SELECT DISTINCT seller_zip_code_prefix FROM raw.sellers", engine)["seller_zip_code_prefix"])
geo_zips = set(geo["geolocation_zip_code_prefix"])

customer_orphans = customer_zips - geo_zips
seller_orphans = seller_zips - geo_zips

print(f"Distinct customer zip prefixes: {len(customer_zips):,}")
print(f"  Not in geolocation: {len(customer_orphans):,} ({len(customer_orphans)/len(customer_zips)*100:.2f}%)")
print()
print(f"Distinct seller zip prefixes: {len(seller_zips):,}")
print(f"  Not in geolocation: {len(seller_orphans):,} ({len(seller_orphans)/len(seller_zips)*100:.2f}%)")

Distinct customer zip prefixes: 14,994
  Not in geolocation: 157 (1.05%)

Distinct seller zip prefixes: 2,246
  Not in geolocation: 7 (0.31%)


In [51]:
customers_unmapped = pd.read_sql("SELECT * FROM raw.customers", engine)
sellers_unmapped = pd.read_sql("SELECT * FROM raw.sellers", engine)

customers_no_geo = customers_unmapped[customers_unmapped["customer_zip_code_prefix"].isin(customer_orphans)]
sellers_no_geo = sellers_unmapped[sellers_unmapped["seller_zip_code_prefix"].isin(seller_orphans)]

print(f"Customer rows affected by missing geolocation: {len(customers_no_geo):,} "
      f"({len(customers_no_geo)/len(customers_unmapped)*100:.2f}% of all customers)")
print(f"Seller rows affected by missing geolocation: {len(sellers_no_geo):,} "
      f"({len(sellers_no_geo)/len(sellers_unmapped)*100:.2f}% of all sellers)")
print()
print("Sample customers without geo:")
print(customers_no_geo[["customer_id", "customer_zip_code_prefix", "customer_city", "customer_state"]].head())
print()
print("Sample sellers without geo:")
print(sellers_no_geo[["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"]].head())

Customer rows affected by missing geolocation: 278 (0.28% of all customers)
Seller rows affected by missing geolocation: 7 (0.23% of all sellers)

Sample customers without geo:
                           customer_id customer_zip_code_prefix customer_city customer_state
354   ecb1725b26e8b8c458181455dfa434ea                    72300      brasilia             DF
382   bcf86029aeed4ed8bac0e16eb14c22f5                    11547       cubatao             SP
877   f4302056f0c58570522590f8181de2c7                    64605         picos             PI
1218  03bbe0ce5c28e05f22917607db798818                    72465      brasilia             DF
1272  ad4950aded55c2ea376be59506456d68                    07729      caieiras             SP

Sample sellers without geo:
                             seller_id seller_zip_code_prefix   seller_city seller_state
473   5962468f885ea01a1b6a97a218797b0a                  82040      curitiba           PR
791   2aafae69bf4c41fbd94053d9413e87ee                  91

### Geolocation — summary

- 1,000,163 rows, no PK (multi-grain on zip prefix). Zero nulls.
- 19,015 distinct zip prefixes; median 29 lat/lng records per zip; max 1,146 (dense urban prefix)
- 8,011 "distinct" city names — but inflated by ~35% due to spelling variants (DQ-017)
- Lat/lng range mostly correct; 42 rows have coordinates outside Brazil (data errors)
- Coverage gap: 1.05% of customer zips, 0.31% of seller zips not in geolocation

**3 data quality findings:**
- **DQ-016:** 42 rows with coordinates outside Brazil's bounding box (data errors landing in Mexico, Spain, etc.)
- **DQ-017:** 45% of zip prefixes have multiple city name variants (uncontrolled free-text input; structurally affects city-based joins)
- **DQ-018:** 1.05% of customer zips / 0.31% of seller zips not covered by geolocation table

Geolocation is the most data-quality-heavy of all 9 tables. Star schema design must use zip prefix (not city name) as the dimensional join key.